In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, ZeroPadding2D, BatchNormalization, Activation, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils import shuffle
import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt
import time
from os import listdir

In [3]:
def load_data(dir_list, image_size):

    X = []
    y = []
    image_width, image_height = image_size

    for directory in dir_list:
        for filename in listdir(directory):

            image = cv2.imread(directory + '/' + filename)
            image = cv2.resize(image, dsize=(image_width, image_height), interpolation=cv2.INTER_CUBIC)
            image = image / 255.
            X.append(image)
            if directory[-3:] == 'ari':
                y.append([1])
            else:
                y.append([0])

    X = np.array(X)
    y = np.array(y)

    X, y = shuffle(X, y)

    print(f'Number of examples is: {len(X)}')
    print(f'X shape is: {X.shape}')
    print(f'y shape is: {y.shape}')

    return X, y

In [12]:
import os

dataset = '/content/drive/MyDrive/AIGENXT/car_dataset_2_class-220709-121158'

dataset_yes = os.path.join(dataset, 'Ferrari')
dataset_no = os.path.join(dataset, 'Honda')

IMG_WIDTH, IMG_HEIGHT = (240, 240)

X, y = load_data([dataset_yes, dataset_no], (IMG_WIDTH, IMG_HEIGHT))

Number of examples is: 647
X shape is: (647, 240, 240, 3)
y shape is: (647, 1)


In [13]:
def split_data(X, y, test_size=0.1):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10)

    return X_train, y_train, X_test, y_test

In [14]:
X_train, y_train, X_test, y_test = split_data(X, y, test_size=0.1)

In [15]:
del X
del y

In [16]:
def build_model(input_shape):

    X_input = Input(input_shape)

    X = ZeroPadding2D((2, 2))(X_input)

    X = Conv2D(32, (5, 5), strides = (1, 1), name = 'conv0')(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((2, 2), name='max_pool0')(X)

    X = Conv2D(32, (5, 5), strides = (1, 1), name = 'conv1')(X)
    X = Activation('relu')(X)

    X = MaxPooling2D((2, 2), name='max_pool1')(X)

    X = Flatten()(X)

    X = Dense(1, activation='sigmoid', name='fc')(X)

    model = Model(inputs = X_input, outputs = X, name='VehicleDetectionModel')

    return model

In [17]:
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, 3)

In [18]:
model = build_model(IMG_SHAPE)

In [19]:
model.summary()

Model: "VehicleDetectionModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 240, 240, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 244, 244, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv0 (Conv2D)                  │ (None, 240, 240, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 240, 240, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool0 (MaxPooling2D)        │ (None, 120, 120, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 116, 116, 32)   │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 116, 116, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool1 (MaxPooling2D)        │ (None, 58, 58, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 107648)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc (Dense)                      │ (None, 1)              │       107,649 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,713 (530.13 KB)

 Trainable params: 135,713 (530.13 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [22]:
filepath="cnn-parameters-improvement-{epoch:02d}-{val_accuracy:.2f}"

checkpoint = ModelCheckpoint("{}.keras".format(filepath), monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

In [23]:
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)#,min_delta=0)

In [24]:
model.fit(x=X_train, y=y_train, batch_size=32, epochs=10, validation_data=(X_test, y_test), callbacks=[checkpoint,callback])

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.5553 - loss: 0.8529
Epoch 1: val_accuracy improved from None to 0.86154, saving model to cnn-parameters-improvement-01-0.86.keras

Epoch 1: finished saving model to cnn-parameters-improvement-01-0.86.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 18s 446ms/step - accuracy: 0.6460 - loss: 0.6983 - val_accuracy: 0.8615 - val_loss: 0.3898
Epoch 2/10
18/19 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8014 - loss: 0.4510
Epoch 2: val_accuracy improved from 0.86154 to 0.87692, saving model to cnn-parameters-improvement-02-0.88.keras

Epoch 2: finished saving model to cnn-parameters-improvement-02-0.88.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - accuracy: 0.8144 - loss: 0.4426 - val_accuracy: 0.8769 - val_loss: 0.3366
Epoch 3/10
18/19 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8591 - loss: 0.3778
Epoch 3: val_accuracy did not improve from 0.87692
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.8643 - loss: 0.3638 - val_accu

In [25]:
def load_data1(file_path,image_size):

    X2 = []

    image_width, image_height = image_size
    print(file_path)
    image = cv2.imread(file_path)
    image = cv2.resize(image, dsize=(image_width, image_height), interpolation=cv2.INTER_CUBIC)
    image = image / 255.
    X2.append(image)

    X2 = np.array(X2)

    print(f'Number of examples is: {len(X2)}')
    print(f'X shape is: {X2.shape}')

    return X2


In [27]:
file_path='/content/drive/MyDrive/AIGENXT/car_dataset_2_class-220709-121158/Honda/010352.jpg'
img=load_data1(file_path,(IMG_WIDTH, IMG_HEIGHT))

/content/drive/MyDrive/AIGENXT/car_dataset_2_class-220709-121158/Honda/010352.jpg
Number of examples is: 1
X shape is: (1, 240, 240, 3)


In [28]:

ss=model.predict(img)

if ss[0][0]>0.5:
    output="Ferrari"
else:
    output="Honda"

print(output)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 696ms/step
Honda


In [30]:
best_model = load_model(filepath='cnn-parameters-improvement-02-0.88.keras')

ss=best_model.predict(img)


if ss[0][0]>0.5:
    output="Ferrari"
else:
    output="Honda"

print(output)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 669ms/step
Honda
